# 📈 Période 2 — Notebook 04 : Évaluation du modèle

**Objectif** : analyse approfondie du meilleur modèle et exemples de recommandations.

Pré-requis : avoir exécuté `03_training_experiment.ipynb`.

In [ ]:
import sys, json, pickle
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score
)
from training.data_loader import load_movies, load_ratings, load_users

# Charger le modèle
with open('../models/naive_bayes_recommender.pkl','rb') as f:
    bundle = pickle.load(f)

model        = bundle['model']
FEATURE_COLS = bundle['feature_columns']

X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

y_pred  = model.predict(X_test[FEATURE_COLS])
y_proba = model.predict_proba(X_test[FEATURE_COLS])

print('Modèle chargé :', bundle.get('run_name','—'))
print('Params :', bundle.get('params'))

In [ ]:
# ── Métriques globales ───────────────────────────────────────────
print('='*50)
print('MÉTRIQUES GLOBALES')
print('='*50)
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba[:,1]):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Non aimé','Aimé']))

cm = confusion_matrix(y_test, y_pred)
print('Matrice de confusion :')
print(f'  TN = {cm[0,0]:5,}  FP = {cm[0,1]:5,}  (films non aimés correctement rejetés vs faux positifs)')
print(f'  FN = {cm[1,0]:5,}  TP = {cm[1,1]:5,}  (films aimés manqués vs correctement recommandés)')

In [ ]:
# ── Analyse par genre ────────────────────────────────────────────
from sklearn.metrics import precision_score

STAT_FEATS = ['user_avg_rating','user_n_ratings','user_std_rating',
              'movie_avg_rating','movie_n_ratings']
CONT_FEATS  = ['Gender_enc','Age','Occupation','Year'] + STAT_FEATS
genre_cols  = [c for c in FEATURE_COLS if c not in CONT_FEATS]

print('Précision par genre (test set) :')
results = []
for genre in genre_cols:
    if genre not in X_test.columns:
        continue
    mask = X_test[genre].values == 1
    if mask.sum() < 30:
        continue
    prec  = precision_score(y_test[mask], y_pred[mask], zero_division=0)
    total = mask.sum()
    results.append((genre, prec, total))

for genre, prec, n in sorted(results, key=lambda x: -x[1]):
    bar = '█' * int(prec * 25)
    print(f'  {genre:20s} {bar:25s}  {prec:.3f}  (n={n})')

In [ ]:
# ── Distribution des scores de confiance ────────────────────────
scores_pos = y_proba[:,1][y_test == 1]
scores_neg = y_proba[:,1][y_test == 0]

print('Distribution des scores P(Liked=1) :')
for label, scores in [('Films aimés   ', scores_pos), ('Films non aimés', scores_neg)]:
    print(f'  {label}  min={scores.min():.3f}  moy={scores.mean():.3f}  max={scores.max():.3f}')

print(f'\nSéparation moyenne : {scores_pos.mean() - scores_neg.mean():.4f}')
print('(plus élevé = meilleure séparation des classes)')

In [ ]:
# ── Recommandations personnalisées ───────────────────────────────
movies  = load_movies()
ratings = load_ratings()
users   = load_users()

movie_stats = (ratings.groupby('MovieID')
               .agg(movie_avg_rating=('Rating','mean'),
                    movie_n_ratings =('Rating','count'))
               .reset_index())
user_stats  = (ratings.groupby('UserID')
               .agg(user_avg_rating =('Rating','mean'),
                    user_n_ratings  =('Rating','count'),
                    user_std_rating =('Rating','std'))
               .reset_index().fillna(0))

genre_cols_m = [c for c in movies.columns if c not in ['MovieID','Title','Genres','Year']]
candidates   = movies[['MovieID','Title','Year'] + genre_cols_m].copy()

profiles = [
    {'Gender_enc':1, 'Age':25, 'Occupation':4,  'label':'👨 Homme 25 ans, ingénieur'},
    {'Gender_enc':0, 'Age':35, 'Occupation':1,  'label':'👩 Femme 35 ans, développeuse'},
    {'Gender_enc':1, 'Age':18, 'Occupation':0,  'label':'👦 Homme 18 ans, étudiant'},
    {'Gender_enc':0, 'Age':50, 'Occupation':7,  'label':'👩 Femme 50 ans, gestionnaire'},
]

print('='*60)
print('🎬 TOP 5 RECOMMANDATIONS PAR PROFIL')
print('='*60)
for profile in profiles:
    label = profile.pop('label')
    recs  = model.recommend_movies(
        profile, candidates,
        user_stats=user_stats, movie_stats=movie_stats,
        top_k=5
    )
    # Joindre les genres
    recs = recs.merge(movies[['MovieID','Genres']], on='MovieID', how='left')
    print(f'\n{label} :')
    for _, r in recs.iterrows():
        print(f'   {r.score:.3f}  {r.Title:40s}  [{r.Genres}]')

## 📝 Conclusions — Période 2

**Ce qui a été accompli :**
1. ✅ Exploration complète des données MovieLens (distribution, biais, densité)
2. ✅ Feature engineering : features utilisateur + film + stats agrégées
3. ✅ 3 runs comparés dans MLflow avec des hyperparamètres différents
4. ✅ Meilleur modèle sauvegardé dans `models/naive_bayes_recommender.pkl`
5. ✅ Recommandations personnalisées fonctionnelles

**Observations :**
- Les stats agrégées (`movie_avg_rating`, `user_avg_rating`) sont les features les plus discriminantes
- Le déséquilibre de classes (~65% positifs) est géré par le seuil de probabilité
- Naïve Bayes offre une bonne interprétabilité et une inférence très rapide

**→ Prochaine étape : Période 3 — Feature Store avec Feast**